# Trabajo de Fin de Carrera (PFC): Evaluación Experimental de Arquitecturas GraphRAG
## Eficiencia de Indexación (E2GraphRAG vs. LightRAG) y Consistencia Bi-temporal (Zep/Graphiti vs. LightRAG)

**Entorno Objetivo**: Google Colab Pro / Google AI Pro (**GPU NVIDIA A100-SXM4-40GB/80GB** o **L4-24GB** + **CPU High-RAM**).

### Estructura de Ejecución Celda por Celda:
- **BLOQUE 1**: Setup del Entorno, Servidor Local `vLLM` (`Llama-3.1-8B-Instruct` / `Qwen2.5-7B-Instruct` en `localhost:8000`), Embeddings `BAAI/bge-m3` y `SpaCy` en CPU.
- **BLOQUE 2**: Experimento 1 — Eficiencia en Indexación (`E2GraphRAG` vs `LightRAG`) sobre particiones escalonadas (`100k`, `250k`, `500k`, `1M` tokens) de `LongMemEval` $\rightarrow$ `experimento1_resultados.csv`.
- **BLOQUE 3**: Experimento 2 — Consistencia Bi-temporal $(t_{VT}, t_{TT})$ (`Zep/Graphiti` vs `LightRAG`) + **Juez LLM (LLM-as-a-Judge)** $\rightarrow$ `experimento2_resultados.csv`.
- **BLOQUE 4**: Visualización Publication-Ready (300 DPI) con `matplotlib` y `seaborn`.

In [ ]:
# =============================================================================
# CELDA 0: Verificación de Hardware en Google Colab Pro (GPU A100/L4 y High-RAM)
# =============================================================================
import multiprocessing
import os
import psutil
import torch

print('=== DIAGNÓSTICO DE HARDWARE EN GOOGLE COLAB PRO ===')
print(f'vCPUs disponibles      : {multiprocessing.cpu_count()} núcleos')
print(f'RAM del Sistema        : {psutil.virtual_memory().total / (1024**3):.2f} GB')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Acelerador GPU Activo  : {gpu_name} ({vram_gb:.2f} GB VRAM)')
else:
    print('ADVERTENCIA: No se detectó GPU. Activa A100 o L4 en Runtime -> Change runtime type.')


In [ ]:
# =============================================================================
# BLOQUE 1 (Parte A): Instalación de Librerías y Modelos SpaCy en CPU
# =============================================================================
!pip install -q lightrag-hku spacy vllm graphiti-core datasets pandas networkx \
    sentence-transformers FlagEmbedding tiktoken openai matplotlib seaborn tqdm nest_asyncio psutil

# Descarga de modelos SpaCy para extracción NER en CPU (Inglés y Español)
!python -m spacy download en_core_web_sm -q
!python -m spacy download es_core_news_sm -q
print('[OK] Librerías y modelos SpaCy instalados correctamente.')

In [ ]:
# =============================================================================
# BLOQUE 1 (Parte B): Inicialización de vLLM (localhost:8000), BGE-M3 y SpaCy
# =============================================================================
import nest_asyncio
nest_asyncio.apply()

# Si usas Google Colab Secrets para HF_TOKEN, descomenta las siguientes 4 líneas:
# try:
#     from google.colab import userdata
#     os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# except Exception:
#     pass

from src.config import ExperimentConfig
from src.inference_setup import LocalInferenceEngine

config = ExperimentConfig()
perfil_hw = config.auto_configure_for_colab_gpu()
print(f'[INFO] Configuración optimizada para: {perfil_hw}')

engine = LocalInferenceEngine(config)

# 1. Iniciar servidor local vLLM en localhost:8000 (Llama-3.1-8B-Instruct / Qwen2.5-7B-Instruct)
engine.start_vllm_server(timeout_seconds=300)

# 2. Cargar modelo de embeddings BAAI/bge-m3 en GPU
engine.load_embedding_model()

# 3. Cargar pipeline SpaCy NER + Sentencizer en CPU
engine.load_spacy_pipeline(lang='en')

In [ ]:
# =============================================================================
# BLOQUE 2: Pipeline de Experimento 1 (Eficiencia en Indexación: E2GraphRAG vs LightRAG)
# Particiones escalonadas de LongMemEval: 100k, 250k, 500k y 1M tokens
# =============================================================================
import asyncio
from src.exp1_indexing import run_experiment_1

df_exp1 = asyncio.run(run_experiment_1(config=config, engine=engine))
display(df_exp1)

In [ ]:
# =============================================================================
# BLOQUE 3: Pipeline de Experimento 2 (Recuperación Bi-temporal: Zep/Graphiti vs LightRAG)
# Ventana Bi-temporal (t_VT, t_TT), Invalidación de Aristas y Juez LLM (LLM-as-a-Judge)
# =============================================================================
import asyncio
from src.exp2_bitemporal import run_experiment_2

df_exp2_detalle, df_exp2_resumen = asyncio.run(run_experiment_2(config=config, engine=engine))
display(df_exp2_resumen)
display(df_exp2_detalle.head(10))

In [ ]:
# =============================================================================
# BLOQUE 4: Visualización y Reporte de Resultados (Gráficos Publication-Ready)
# =============================================================================
from IPython.display import Image, display
from src.visualization import generate_all_figures

fig1_path, fig2_path = generate_all_figures(config=config)
display(Image(filename=str(fig1_path)))
display(Image(filename=str(fig2_path)))